# SteelFlow: commercial and supply-chain domain EDA

This notebook broadens the EDA beyond forecasting and delivery. Every section starts from a business grain, computes operational KPIs, and identifies useful prediction targets.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
con = duckdb.connect()
sns.set_theme(style="whitegrid")


## 1. Carrier and route performance

Grain: one shipment or one carrier-route-month. We measure volume, lateness, cost, and service consistency.

In [ ]:
ship = con.execute(f"SELECT * FROM read_parquet('{(DATA/'outbound_shipments.parquet').as_posix()}')").fetchdf()
ship['dispatch_date'] = pd.to_datetime(ship['dispatch_date'])
carrier_perf = ship.groupby('carrier_id').agg(shipments=('shipment_id','size'), weight_kg=('shipment_weight_kg','sum'), late_rate=('is_late','mean'), avg_delay_hours=('delay_hours_vs_promised','mean'), freight_per_kg=('quoted_freight_cost_inr','sum'))
carrier_perf['freight_per_kg'] = carrier_perf['freight_per_kg'] / carrier_perf['weight_kg'].clip(lower=1)
carrier_perf.sort_values(['late_rate','shipments'], ascending=[False,False]).head(20)

In [ ]:
route_perf = ship.groupby(['route_id','route_type']).agg(shipments=('shipment_id','size'), late_rate=('is_late','mean'), avg_transit_hours=('actual_transit_hours','mean'), avg_cost_per_kg=('quoted_freight_cost_inr', lambda s: s.sum()))
route_perf.sort_values('late_rate', ascending=False).head(20)

## 2. Receivables, payment habits, and credit risk

Grain: customer-month for behaviour and invoice for exposure. Candidate targets include overdue/default events, payment delay, and customer credit deterioration.

In [ ]:
ar = con.execute(f"SELECT * FROM read_parquet('{(DATA/'accounts_receivable.parquet').as_posix()}')").fetchdf()
print(ar.columns.tolist())
customer_behaviour = con.execute(f"SELECT * FROM read_parquet('{(DATA/'customer_payment_behaviour_monthly.parquet').as_posix()}')").fetchdf()
customer_behaviour.head()

In [ ]:
# Inspect payment metrics and build a simple risk score from available behaviour fields
customer_behaviour.describe(include='all').T.head(30)

In [ ]:
credit = con.execute(f"SELECT * FROM read_parquet('{(DATA/'customer_credit_state.parquet').as_posix()}')").fetchdf()
credit.columns.tolist()

## 3. Customer segments, habits, and churn-risk features

Churn is not directly observed unless a future inactivity or default outcome exists. We therefore create an EDA proxy: recency, order frequency, spend/weight trend, cancellations, payment delay, disputes, and credit-state deterioration. The modeling target should be defined as inactivity in a future window, then validated against the synthetic business rules.

In [ ]:
orders = con.execute(f"SELECT * FROM read_parquet('{(DATA/'customer_orders.parquet').as_posix()}')").fetchdf()
orders['order_date'] = pd.to_datetime(orders['order_date'])
customer_rfm = orders.groupby('customer_id').agg(last_order=('order_date','max'), orders=('order_id','nunique'), revenue=('order_value_ex_gst','sum'), weight_kg=('ordered_weight_kg','sum'), avg_order_value=('order_value_ex_gst','mean'), avg_payment_delay=('expected_payment_delay_days_at_check','mean'))
reference_date = orders['order_date'].max() + pd.Timedelta(days=1)
customer_rfm['recency_days'] = (reference_date - customer_rfm['last_order']).dt.days
customer_rfm.sort_values('recency_days', ascending=False).head(20)

In [ ]:
customer_rfm[['recency_days','orders','revenue','weight_kg','avg_order_value','avg_payment_delay']].describe(percentiles=[.5,.9,.95,.99])

## 4. Inventory, suppliers, and warehouses

Inventory EDA should separate stock position from movements, then measure stockouts, turnover, coverage, and supplier concentration. These support reorder-demand, stockout-risk, supplier-delay, and warehouse-capacity models.

In [ ]:
fg = con.execute(f"SELECT * FROM read_parquet('{(DATA/'finished_goods_inventory_snapshots.parquet').as_posix()}')").fetchdf()
raw = con.execute(f"SELECT * FROM read_parquet('{(DATA/'raw_material_inventory_snapshots.parquet').as_posix()}')").fetchdf()
wh = con.execute(f"SELECT * FROM read_parquet('{(DATA/'warehouse_stock_snapshots.parquet').as_posix()}')").fetchdf()
print('finished goods', fg.shape, fg.columns.tolist())
print('raw material', raw.shape, raw.columns.tolist())
print('warehouse', wh.shape, wh.columns.tolist())

In [ ]:
sup = con.execute(f"SELECT * FROM read_parquet('{(DATA/'supplier_material_month_performance.parquet').as_posix()}')").fetchdf()
print(sup.columns.tolist())
sup.head()

In [ ]:
# Concentration and service-risk view
if 'supplier_id' in sup and 'material_id' in sup:
    print(sup.groupby('supplier_id').size().sort_values(ascending=False).head(15))
if 'warehouse_id' in wh:
    display(wh.groupby('warehouse_id').size().sort_values(ascending=False).head(15).to_frame('snapshot_rows'))

## EDA handoff

The next modeling stage can use: carrier late-risk, freight-cost estimation, customer future-inactivity risk, payment/default risk, stockout risk, supplier performance, and warehouse inventory forecasting. Before training, confirm each target is defined after the feature cutoff and split all data chronologically.